# Rankings Refresh Pipeline

In [27]:
# Load common data science and notebook packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import json
sys.path.append(os.path.dirname(os.path.abspath('')))
from scripts.load_data import load_data
from scripts.update_player_key import update_player_key_dict

# Display settings for pandas
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

# Enable inline plotting for Jupyter
%matplotlib inline

# Explain: Loaded pandas, numpy, matplotlib, seaborn for data analysis and visualization.


In [28]:
# Load each file in data_path into a distinct DataFrame using load_data.

data_path = "../data/rankings current/update/"

# List all files in the data_path directory
files = [f for f in os.listdir(data_path) if not f.startswith('.')]

# Sort files for reproducibility (optional, but helps with mapping)
files.sort()

# Check that there are at least three files
if len(files) < 3:
    raise ValueError("Expected at least three files in the data_path directory.")

# Load each file into a DataFrame using load_data
# Step 1: Create a lookup table mapping comment names to generalized file names.
# We'll use the first 12 characters of each file name (adjust as needed for your files).
# The keys are the comment names, the values are the generalized file names.

lookup_keys = [
    "fpts",
    "fantasypros",
    "jj",
    "draftshark_adp",
    "draftshark_rank",
    "hayden_winks"
]

# Generalize file names by taking the first 12 characters (adjust if needed)
generalized_files = [f[:11] for f in files[:len(lookup_keys)]]

# Build the lookup dictionary
file_lookup = dict(zip(lookup_keys, generalized_files))

# Step 2: Use the lookup to read in each file and assign to a variable named after the key.
# We'll use globals() to dynamically assign variable names.

# Create a dictionary of DataFrames, keyed by lookup_key, with values from load_data
dataframes = {}
for key, gen_file in file_lookup.items():
    # Find the actual file in files that starts with gen_file
    matched_file = next((f for f in files if f.startswith(gen_file)), None)
    if matched_file:
        dataframes[key] = load_data(os.path.join(data_path, matched_file))
    else:
        raise ValueError(f"No file found for key '{key}' with prefix '{gen_file}'.")

# Now you have a dictionary: dataframes['fpts'], dataframes['fantasypros'], etc.

In [29]:
file_lookup

{'fpts': '2025 NFL Fa',
 'fantasypros': 'FantasyPros',
 'jj': 'Redraft1QB_',
 'draftshark_adp': 'player_adps',
 'draftshark_rank': 'preseason_r',
 'hayden_winks': 'tableDownlo'}

In [30]:
# update col names in dataframes dict

from scripts.clean_cols import cols_dict

for key, df in dataframes.items():
    if key in cols_dict:
        df.columns = cols_dict[key]


In [31]:
# for each dataframe, compare the values in PLAYER NAME to the values in player_key_dict.json. if there is a match, join the player "key" to the dataframe as "PLAYER ID"

# Load the player key dictionary
with open('../player_key_dict.json', 'r') as f:
    player_key_dict = json.load(f)

# Create reverse mapping from player names to keys
player_name_to_key = {}
for key, value in player_key_dict.items():
    if isinstance(value, list):
        # Handle cases where multiple names map to same key
        for name in value:
            player_name_to_key[name] = key
    else:
        # Handle single name mapping
        player_name_to_key[value] = key

# For each dataframe, add PLAYER ID column based on name matching
for key, df in dataframes.items():
    # Create PLAYER ID column by mapping player names to keys
    df['PLAYER ID'] = df['PLAYER NAME'].map(player_name_to_key)
    
    # Print summary of matches for debugging
    total_players = len(df)
    matched_players = df['PLAYER ID'].notna().sum()
    print(f"{key}: {matched_players}/{total_players} players matched ({matched_players/total_players*100:.1f}%)")


fpts: 483/527 players matched (91.7%)
fantasypros: 389/494 players matched (78.7%)
jj: 243/250 players matched (97.2%)
draftshark_adp: 239/332 players matched (72.0%)
draftshark_rank: 438/515 players matched (85.0%)
hayden_winks: 247/267 players matched (92.5%)


In [32]:
#### Process FPTS data ####

# Create VBD (Value Based Drafting) field for fpts dataframe
# VBD = Player's FPTS - Baseline FPTS for their position
# Baselines: QB=12, RB=24, WR=30, TE=12

# Define baseline values for each position
baseline_dict = {
    'QB': 6,
    'RB': 24, 
    'WR': 30,
    'TE': 12
}

# Create VBD column by finding the baseline player's FPTS for each position and subtracting from player's FPTS
def calculate_vbd(row):
    if pd.isna(row['FPTS']) or row['POS'] not in baseline_dict:
        return None
    
    pos = row['POS']
    baseline_rank = baseline_dict[pos]
    
    # Get all players of the same position, sorted by FPTS descending
    pos_players = dataframes['fpts'][dataframes['fpts']['POS'] == pos].sort_values('FPTS', ascending=False)
    
    # Find the FPTS of the player at the baseline rank position
    if len(pos_players) >= baseline_rank:
        baseline_fpts = pos_players.iloc[baseline_rank - 1]['FPTS']  # -1 because iloc is 0-indexed
    else:
        # If not enough players at position, use 0 as baseline
        baseline_fpts = 0
    
    return row['FPTS'] - baseline_fpts

dataframes['fpts']['VBD'] = dataframes['fpts'].apply(calculate_vbd, axis=1)

# Reduce VBD for QB position
qb_adjustment = 0.50
dataframes['fpts'].loc[dataframes['fpts']['POS'] == 'QB', 'VBD'] = dataframes['fpts'].loc[dataframes['fpts']['POS'] == 'QB', 'VBD'] * qb_adjustment


# Create RK (ranking) column based on VBD values (highest VBD = rank 1)
dataframes['fpts']['RK'] = dataframes['fpts']['VBD'].rank(ascending=False, method='min')
# Only apply minimum RK of 24 for QB position
# dataframes['fpts'].loc[dataframes['fpts']['POS'] == 'QB', 'RK'] = dataframes['fpts'].loc[dataframes['fpts']['POS'] == 'QB', 'RK'].clip(lower=24)

# Create POS RANK column that ranks players within their position based on RK
dataframes['fpts']['POS RANK'] = dataframes['fpts'].groupby('POS')['RK'].rank(method='min')

# Display first few rows to verify results
print("FPTS dataframe with VBD, RK, and POS RANK columns:")
print(dataframes['fpts'][['PLAYER NAME', 'POS', 'FPTS', 'VBD', 'RK', 'POS RANK']].sort_values('RK').head(10))
# Print baseline players for each position
print("Baseline players for each position:")
print("=" * 50)

for pos, baseline_rank in baseline_dict.items():
    # Get all players of the same position, sorted by FPTS descending
    pos_players = dataframes['fpts'][dataframes['fpts']['POS'] == pos].sort_values('FPTS', ascending=False)
    
    if len(pos_players) >= baseline_rank:
        baseline_player = pos_players.iloc[baseline_rank - 1]  # -1 because iloc is 0-indexed
        print(f"{pos} Baseline (Rank {baseline_rank}): {baseline_player['PLAYER NAME']} - {baseline_player['FPTS']} FPTS")
    else:
        print(f"{pos} Baseline (Rank {baseline_rank}): Not enough players at position - 0 FPTS")


# using "FPTS" column, create a VBD field (12, 24, 30, 12). then, create a column "RK" which is the ranking based on VBD.
# create column "POS RANK" that ranks players based on "RK" as a subset of "POS" (eg, ranks each QB relative to other QBs) 

FPTS dataframe with VBD, RK, and POS RANK columns:
            PLAYER NAME POS   FPTS    VBD    RK  POS RANK
5        Saquon Barkley  RB  297.8  117.4   1.0       1.0
18        Ja'Marr Chase  WR  263.4   98.4   2.0       1.0
8        Bijan Robinson  RB  278.7   98.3   3.0       2.0
7          Jahmyr Gibbs  RB  278.7   98.3   3.0       2.0
22           Puka Nacua  WR  258.7   93.7   5.0       2.0
28         Malik Nabers  WR  254.8   89.8   6.0       3.0
17        Derrick Henry  RB  264.5   84.1   7.0       4.0
21        Ashton Jeanty  RB  259.0   78.6   8.0       5.0
29  Christian McCaffrey  RB  253.2   72.8   9.0       6.0
36     Justin Jefferson  WR  237.6   72.6  10.0       4.0
Baseline players for each position:
QB Baseline (Rank 6): Patrick Mahomes - 280.2 FPTS
RB Baseline (Rank 24): Quinshon Judkins - 180.4 FPTS
WR Baseline (Rank 30): Chris Godwin - 165.0 FPTS
TE Baseline (Rank 12): Evan Engram - 119.6 FPTS


In [33]:
#### Processfantasy_pros ####

# Create POS RANK column that ranks players within their position based on RK

# Remove integers from POS column values in fantasypros dataframe
dataframes['fantasypros']['POS'] = dataframes['fantasypros']['POS'].str.replace(r'\d+', '', regex=True)
dataframes['fantasypros']['POS RANK'] = dataframes['fantasypros'].groupby('POS')['RK'].rank(method='min') 

# create column "POS RANK" that ranks players based on "RK" as a subset of "POS" (eg, ranks each QB relative to other QBs)

In [34]:
dataframes['fantasypros'].groupby('POS')['RK'].mean()

POS
DST    258.000000
K      291.827586
QB     235.481481
RB     229.224638
TE     290.697368
WR     237.611765
Name: RK, dtype: float64

In [35]:
dataframes['fantasypros']['POS RANK']

0        1.0
1        1.0
2        2.0
3        2.0
4        3.0
       ...  
489    138.0
490     75.0
491    169.0
492    170.0
493     76.0
Name: POS RANK, Length: 494, dtype: float64

In [36]:
# draftshark ADP

# Convert SLEEPER ADP to string and split into round and pick
dataframes['draftshark_adp']['SLEEPER ADP'] = dataframes['draftshark_adp']['SLEEPER ADP'].astype(str)

# Create ADP ROUND column (value before the decimal point)
dataframes['draftshark_adp']['ADP ROUND'] = dataframes['draftshark_adp']['SLEEPER ADP'].str.split('.').str[0].astype(int)

# Create ADP ROUND PICK column (value after the decimal point, between 1-12)
dataframes['draftshark_adp']['ADP ROUND PICK'] = dataframes['draftshark_adp'].index + 1 
# Update ADP ROUND PICK to be divided by (ADP ROUND - 1)
dataframes['draftshark_adp']['ADP ROUND PICK'] = dataframes['draftshark_adp']['ADP ROUND PICK'] - ((dataframes['draftshark_adp']['ADP ROUND'] - 1) * 12)


# Create ADP RANK column using the formula: ((ADP ROUND - 1) * 12) + ADP ROUND PICK
dataframes['draftshark_adp']['ADP RANK'] = dataframes['draftshark_adp'].index + 1

# Display first few rows to verify results
print("DraftShark ADP dataframe with new columns:")
print(dataframes['draftshark_adp'][['PLAYER NAME', 'SLEEPER ADP', 'ADP ROUND', 'ADP ROUND PICK', 'ADP RANK']].head(14))


# create a column "ADP ROUND" that is equal to the value before the "." in "SLEEPER ADP". this may require converting "SLEEPER ADP" values to string.
# create a column "ADP ROUND PICK" that is equal to the value after the "." in "SLEEPER ADP". this may require converting "SLEEPER ADP" values to string. this should be a value between 1 and 12.
# create a column "ADP RANK" which is equal to (("ADP ROUND" - 1) * 12) + "ADP ROUND PICK"
#  

DraftShark ADP dataframe with new columns:
          PLAYER NAME SLEEPER ADP  ADP ROUND  ADP ROUND PICK  ADP RANK
0       Ja'Marr Chase         1.1          1               1         1
1      Saquon Barkley         1.2          1               2         2
2    Justin Jefferson         1.3          1               3         3
3      Bijan Robinson         1.4          1               4         4
4        Jahmyr Gibbs         1.5          1               5         5
5         CeeDee Lamb         1.6          1               6         6
6   Amon-Ra St. Brown         1.7          1               7         7
7          Puka Nacua         1.8          1               8         8
8        Malik Nabers         1.9          1               9         9
9        Nico Collins         1.1          1              10        10
10   Brian Thomas Jr.        1.11          1              11        11
11      Derrick Henry        1.12          1              12        12
12       Brock Bowers         2.1 

In [37]:
dataframes['draftshark_rank'].head()

,PLAYER NAME,POS,TEAM,G,ADP,BYE,SOS,INJURY RISK,FLOOR PROJ,CONS PROJ,DS PROJ,CEILING PROJ,3D VALUE,PLAYER ID
0,Ja'Marr Chase,WR,CIN,17,1.01,10,2%,75%,238,268,287,333,100,NaN
1,Saquon Barkley,RB,PHI,17,1.04,9,-25%,57%,250,283,301,346,98,BarkSa00
2,Bijan Robinson,RB,ATL,17,1.02,5,-1%,42%,250,279,298,340,96,RobiBi01
3,CeeDee Lamb,WR,DAL,17,1.05,10,8%,50%,225,235,271,311,88,LambCe00
4,Justin Jefferson,WR,MIN,17,1.03,6,17%,67%,230,250,261,322,87,JeffJu00


In [38]:
#### Process draftsharks rank ####

# Create new "RK" column that ranks players based on "3D Value" (inverse ranking)
# Higher 3D Value gets lower rank number (rank 1 for highest value)
dataframes['draftshark_rank']['RK'] = dataframes['draftshark_rank'].index + 1

# Create "POS RANK" column that ranks players within their position based on RK
dataframes['draftshark_rank']['POS RANK'] = dataframes['draftshark_rank'].groupby('POS')['RK'].rank(method='min')

# Display first few rows to verify results
print("DraftShark Rank dataframe with new columns:")
print(dataframes['draftshark_rank'][['PLAYER NAME', 'POS', '3D VALUE', 'RK', 'POS RANK']].head(10))


# drop the "RK" column
# create column "RK" that ranks players based on the "3D Value" column. The "RK" should be the inverse of 3D Value (ie, the highest 3D value number should be 1, the second highest 2, etc)
# create column "POS RANK" that ranks players based on "RK" as a subset of "POS" (eg, ranks each QB relative to other QBs)

DraftShark Rank dataframe with new columns:
           PLAYER NAME POS  3D VALUE  RK  POS RANK
0        Ja'Marr Chase  WR       100   1       1.0
1       Saquon Barkley  RB        98   2       1.0
2       Bijan Robinson  RB        96   3       2.0
3          CeeDee Lamb  WR        88   4       2.0
4     Justin Jefferson  WR        87   5       3.0
5  Christian McCaffrey  RB        86   6       3.0
6           Puka Nacua  WR        84   7       4.0
7         Jahmyr Gibbs  RB        83   8       4.0
8        Derrick Henry  RB        82   9       5.0
9      Jonathan Taylor  RB        79  10       6.0


In [39]:
# Create df_rank dataframe starting with player keys
df_rank = pd.DataFrame({'PLAYER ID': list(player_key_dict.keys())})

# Merge with fpts dataframe to get PLAYER NAME and POS
df_rank = df_rank.merge(
    dataframes['fpts'][['PLAYER ID', 'PLAYER NAME', 'POS']], 
    on='PLAYER ID', 
    how='right'
)


In [40]:

# Create df_rank dataframe starting with player keys
df_rank = pd.DataFrame({'PLAYER ID': list(player_key_dict.keys())})

# Merge with fpts dataframe to get PLAYER NAME and POS
df_rank = df_rank.merge(
    dataframes['fantasypros'][['PLAYER ID', 'PLAYER NAME', 'POS']], 
    on='PLAYER ID', 
    how='left'
)

# Left join draftshark_adp dataframe to df_rank on PLAYER ID
df_rank = df_rank.merge(
    dataframes['draftshark_adp'][['PLAYER ID', 'ADP ROUND', 'ADP ROUND PICK', 'ADP RANK']],
    on='PLAYER ID',
    how='left'
)

# Cycle through each dataframe and left join relevant columns
for key, df in dataframes.items():
    # Define columns to look for in each dataframe
    columns_to_join = []
    
    # Check for RK column
    if 'RK' in df.columns:
        columns_to_join.append('RK')
    
    # Check for POS RANK column  
    if 'POS RANK' in df.columns:
        columns_to_join.append('POS RANK')
    
    # Check for TIER column
    if 'TIER' in df.columns:
        columns_to_join.append('TIER')
    
    # If we found any relevant columns, perform the join
    if columns_to_join:
        # Add PLAYER ID to columns to join
        join_columns = ['PLAYER ID'] + columns_to_join
        
        # Perform left join with prefix
        df_rank = df_rank.merge(
            df[join_columns],
            on='PLAYER ID',
            how='left',
            suffixes=('', f'_{key}')
        )
        
        # Rename columns to add prefix (except PLAYER ID)
        rename_dict = {}
        for col in columns_to_join:
            rename_dict[col] = f'{key}_{col}'
        
        df_rank = df_rank.rename(columns=rename_dict)



# Reorder columns to group RK and POS RANK columns together
# Get all column names
all_columns = list(df_rank.columns)

# Separate columns by type
adp_columns = [col for col in all_columns if 'ADP' in col]
rk_columns = [col for col in all_columns if 'RK' in col and 'POS' not in col]
pos_rank_columns = [col for col in all_columns if 'POS RANK' in col]
tier_columns = [col for col in all_columns if 'TIER' in col]
other_columns = [col for col in all_columns if col not in rk_columns and col not in pos_rank_columns and col not in tier_columns and col not in adp_columns]

# Reorder columns: other columns first, then RK columns, then POS RANK columns
reordered_columns = other_columns + adp_columns + rk_columns + pos_rank_columns + tier_columns

# Apply the new column order
df_rank = df_rank[reordered_columns]


# Display first few rows to verify results
print("df_rank dataframe created:")
print(df_rank.head(10))
print(f"\nTotal rows: {len(df_rank)}")
print(f"Columns: {list(df_rank.columns)}")



df_rank dataframe created:
  PLAYER ID          PLAYER NAME  POS  ADP ROUND  ADP ROUND PICK  ADP RANK  fpts_RK  fantasypros_RK  jj_RK  \
0  McCaCh01  Christian McCaffrey   RB        2.0             6.0      18.0      9.0            14.0    9.0   
1  JackLa00        Lamar Jackson   QB        3.0             2.0      26.0     39.0            26.0   26.0   
2  HenrDe00        Derrick Henry   RB        1.0            12.0      12.0      7.0            12.0   15.0   
3  JoneAa00                  NaN  NaN        6.0            14.0      74.0     62.0             NaN   62.0   
4  ElliEz00      Ezekiel Elliott   RB        NaN             NaN       NaN      NaN           462.0    NaN   
5  CookDa01                  NaN  NaN        NaN             NaN       NaN      NaN             NaN    NaN   
6  ThomMi05                  NaN  NaN        NaN             NaN       NaN      NaN             NaN    NaN   
7  KelcTr00         Travis Kelce   TE        6.0            18.0      78.0     41.0          

In [41]:
# implied round and ranking? 
# adp delta 
# average 
# standard dev


In [42]:
# drop na
df_rank = df_rank.dropna(subset=['PLAYER NAME', 'ADP ROUND'])

In [43]:
# keep only main positions
df_rank = df_rank[df_rank['POS'].isin(['WR', 'RB', 'QB', 'TE'])]

In [48]:
df_rank[df_rank['PLAYER NAME']=="Jamarr Chase"]

,PLAYER ID,PLAYER NAME,POS,ADP ROUND,ADP ROUND PICK,ADP RANK,fpts_RK,fantasypros_RK,jj_RK,draftshark_rank_RK,hayden_winks_RK,fpts_POS RANK,fantasypros_POS RANK,jj_POS RANK,draftshark_rank_POS RANK,hayden_winks_POS RANK,fpts_TIER,fantasypros_TIER,jj_TIER


In [45]:
df_rank.to_csv('df_rank_clean.csv', index=False)